# 01 — Data Ingestion & Lake Setup

Pull ENTSO-E day-ahead prices + grid load (DE_LU zone) and Open-Meteo weather data.
Verify the hive-partitioned Parquet structure.

**Prerequisites:** `.env` file with `ENTSOE_API_KEY` set.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../.env')

BASE_DIR = Path('..').resolve()
print('Project root:', BASE_DIR)
print('ENTSOE_API_KEY set:', bool(os.environ.get('ENTSOE_API_KEY')))

## 1.1 Collect data (3-month sample)

In [ ]:
from etl.collect import CollectPrices, CollectLoad, CollectWeather
from datetime import datetime, timezone

api_key = os.environ['ENTSOE_API_KEY']
start = datetime(2023, 1, 1, tzinfo=timezone.utc)
end   = datetime(2023, 3, 31, tzinfo=timezone.utc)

CollectPrices(api_key).process(start, end)
CollectLoad(api_key).process(start, end)
CollectWeather().process(start, end)

## 1.2 Verify partition structure

In [ ]:
import duckdb

con = duckdb.connect()
for source in ['prices', 'load', 'weather']:
    pattern = f'../data/raw/{source}/**/*.parquet'
    try:
        r = con.execute(f"SELECT COUNT(*), MIN(timestamp_utc), MAX(timestamp_utc) FROM read_parquet('{pattern}', hive_partitioning=true)").fetchone()
        print(f'{source:10s}: {r[0]:>6,} rows | {r[1]} → {r[2]}')
    except Exception as e:
        print(f'{source}: ERROR — {e}')

## 1.3 Schema inspection

In [ ]:
for source in ['prices', 'load', 'weather']:
    pattern = f'../data/raw/{source}/**/*.parquet'
    try:
        df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{pattern}', hive_partitioning=true) LIMIT 1").fetchdf()
        print(f'\n--- {source} schema ---')
        print(df[['column_name','column_type']].to_string(index=False))
    except Exception as e:
        print(f'{source}: {e}')

## 1.4 Missing data heatmap (prices)

In [ ]:
import pandas as pd
import plotly.express as px

df_prices = con.execute(
    "SELECT * FROM read_parquet('../data/raw/prices/**/*.parquet', hive_partitioning=true) ORDER BY timestamp_utc"
).fetchdf()

# Create full hourly spine and identify gaps
spine = pd.date_range(df_prices.timestamp_utc.min(), df_prices.timestamp_utc.max(), freq='h', tz='UTC')
df_spine = pd.DataFrame({'timestamp_utc': spine})
df_merged = df_spine.merge(df_prices[['timestamp_utc']], on='timestamp_utc', how='left', indicator=True)
n_missing = (df_merged['_merge'] == 'left_only').sum()
print(f'Missing hours: {n_missing} / {len(spine)} ({n_missing/len(spine)*100:.2f}%)')

# Sample price plot
fig = px.line(df_prices.head(2000), x='timestamp_utc', y='price_eur_mwh',
              title='Raw Prices — First 2000 hours', template='plotly_dark')
fig.show()